In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('data/greenroom.db')

# List all tables
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;",
    conn
)
print(tables)

            name
0       agencies
1         agents
2        artists
3          comps
4          deals
5       expenses
6    settlements
7          shows
8   ticket_sales
9          users
10        venues


In [2]:
for table in tables['name']:
    schema = pd.read_sql_query(f"PRAGMA table_info({table});", conn)
    print(f"\n=== {table} ===")
    print(schema[['name', 'type']].to_string(index=False))


=== agencies ===
name type
  id TEXT
name TEXT

=== agents ===
             name type
               id TEXT
             name TEXT
        agency_id TEXT
            email TEXT
            phone TEXT
preferences_notes TEXT

=== artists ===
            name    type
              id    TEXT
            name    TEXT
        agent_id    TEXT
   manager_email    TEXT
           genre    TEXT
prior_show_count INTEGER

=== comps ===
               name    type
                 id    TEXT
            show_id    TEXT
           category    TEXT
              count INTEGER
         face_value    REAL
counts_toward_gross INTEGER
              notes    TEXT

=== deals ===
               name    type
                 id    TEXT
            show_id    TEXT
          deal_type    TEXT
   guarantee_amount    REAL
         percentage    REAL
   percentage_basis    TEXT
        expense_cap    REAL
    hospitality_cap    REAL
       bonuses_json    TEXT
deal_notes_freetext    TEXT
         created_at I

In [3]:
for table in tables['name']:
    count = pd.read_sql_query(f"SELECT COUNT(*) as n FROM {table};", conn).iloc[0]['n']
    print(f"{table}: {count} rows")

agencies: 5 rows
agents: 14 rows
artists: 59 rows
comps: 1935 rows
deals: 537 rows
expenses: 2943 rows
settlements: 537 rows
shows: 537 rows
ticket_sales: 537 rows
users: 2 rows
venues: 1 rows


In [4]:
pd.read_sql_query("""
    SELECT id, show_id, status, signoff_text, notes
    FROM settlements
    WHERE status = 'disputed' 
      AND signoff_text IS NOT NULL
      AND signoff_text != ''
    LIMIT 20;
""", conn)

,id,show_id,status,signoff_text,notes
0,stl_show_0007,show_0007,disputed,Looks good — TM. Wire to the usual account whe...,"[Mariana, internal] TM signed off Sunday morni..."
1,stl_show_0015,show_0015,disputed,OK. Good night.,None
2,stl_show_0036,show_0036,disputed,Looks good.,None
3,stl_show_0038,show_0038,disputed,OK. Good night.,None
4,stl_show_0044,show_0044,disputed,OK. Good night.,None
5,stl_show_0064,show_0064,disputed,👍,None
6,stl_show_0069,show_0069,disputed,👍,None
7,stl_show_0142,show_0142,disputed,ok wire monday,None
8,stl_show_0152,show_0152,disputed,Looks good.,None
9,stl_show_0159,show_0159,disputed,👍,Marketing recoup pre-deducted from gross.


In [5]:
pd.read_sql_query("""
    SELECT status, COUNT(*) as n
    FROM settlements
    GROUP BY status
    ORDER BY n DESC;
""", conn)

,status,n
0,paid,447
1,finalized,32
2,disputed,24
3,signed,12
4,submitted,8
5,in_review,7
6,voided,4
7,revised,2
8,draft,1


In [6]:
pd.read_sql_query("""
    SELECT deal_type, COUNT(*) as n, 
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) as pct
    FROM deals
    GROUP BY deal_type
    ORDER BY n DESC;
""", conn)

,deal_type,n,pct
0,vs,195,36.3
1,flat,185,34.5
2,percentage_of_net,109,20.3
3,door,30,5.6
4,percentage_of_gross,18,3.4


In [7]:
sample = pd.read_sql_query("""
    SELECT id, deal_type, guarantee_amount, percentage, expense_cap, deal_notes_freetext
    FROM deals
    WHERE deal_notes_freetext IS NOT NULL
      AND LENGTH(deal_notes_freetext) > 50
    ORDER BY RANDOM()
    LIMIT 10;
""", conn)

# Print each row's notes so we can actually read them
for _, row in sample.iterrows():
    print(f"--- Deal {row['id'][:8]} ({row['deal_type']}) ---")
    print(f"  Structured: guarantee={row['guarantee_amount']}, pct={row['percentage']}, cap={row['expense_cap']}")
    print(f"  Notes: {row['deal_notes_freetext']}")
    print()

--- Deal deal_sho (vs) ---
  Structured: guarantee=1383.0, pct=0.9, cap=700.0
  Notes: $1,383 guarantee vs 90% of net after expenses, whichever greater. Expenses capped $700. Hospitality cap $500.

--- Deal deal_sho (door) ---
  Structured: guarantee=nan, pct=nan, cap=1350.0
  Notes: Door deal. Artist gets ticket revenue minus expenses (capped $1350). DIY/experimental tour.

--- Deal deal_sho (percentage_of_net) ---
  Structured: guarantee=nan, pct=0.85, cap=950.0
  Notes: 85% of net after expenses. Expenses capped $950. No guarantee.

--- Deal deal_sho (vs) ---
  Structured: guarantee=2324.0, pct=0.85, cap=1150.0
  Notes: Deal: $2,324 vs 85/15 after expenses. Expense cap 1150, hospitality cap 600.

--- Deal deal_sho (percentage_of_net) ---
  Structured: guarantee=nan, pct=0.85, cap=250.0
  Notes: 85% of net after expenses. Expenses capped $250. No guarantee.

--- Deal deal_sho (vs) ---
  Structured: guarantee=4856.0, pct=0.85, cap=2450.0
  Notes: $4,856 guarantee vs 85% of net after e

In [8]:
import json

settlements_df = pd.read_sql_query("""
    SELECT id, show_id, status, recoups_json
    FROM settlements
    WHERE recoups_json IS NOT NULL AND recoups_json != ''
""", conn)

recoup_rows = []
for _, row in settlements_df.iterrows():
    try:
        recoups = json.loads(row['recoups_json'])
        for r in recoups:
            recoup_rows.append({
                'settlement_id': row['id'],
                'settlement_status': row['status'],
                'category': r.get('category'),
                'amount': r.get('amount'),
                'recoup_status': r.get('status'),
            })
    except:
        pass

recoups_df = pd.DataFrame(recoup_rows)
print(f"Total recoup line items: {len(recoups_df)}")
print("\nDispute rate by category:")
print(recoups_df.groupby('category')['recoup_status'].value_counts(normalize=True).unstack())

Total recoup line items: 103

Dispute rate by category:
recoup_status          agreed  disputed
category                               
hospitality_overage  0.642857  0.357143
marketing            0.807018  0.192982
prior_advance        1.000000       NaN
production_overage   0.636364  0.363636


In [9]:
pd.read_sql_query("""
    SELECT ag.name as agency, 
           s.status as settlement_status,
           COUNT(*) as n
    FROM settlements s
    JOIN shows sh ON s.show_id = sh.id
    JOIN artists ar ON sh.artist_id = ar.id
    JOIN agents a ON ar.agent_id = a.id
    JOIN agencies ag ON a.agency_id = ag.id
    GROUP BY ag.name, s.status
    ORDER BY ag.name, n DESC;
""", conn)

,agency,settlement_status,n
0,CAA,paid,43
1,CAA,disputed,2
2,CAA,in_review,1
3,CAA,finalized,1
4,CAA,draft,1
5,Independent,paid,242
6,Independent,finalized,19
7,Independent,signed,9
8,Independent,disputed,9
9,Independent,submitted,6
